# VNExpress Recommendation - Benchmark Suite (CB vs CF)

This notebook reproduces the complete experimental results for the final report.
It is divided into two main theoretical branches:
1. **Content-Based (CB)**: Semantic matching using NLP embeddings.
2. **Collaborative Filtering (CF)**: Behavioral matching using GNNs & Contrastive Learning.

In [ ]:
# 1. Environment Setup
!pip install -q torch-geometric transformers pandas numpy scipy pyyaml tabulate tqdm sentence-transformers

from kaggle_secrets import UserSecretsClient
import os
REPO_OWNER = "GadGadGad"
REPO_NAME = "DS300-Final-Project"

try:
    github_token = UserSecretsClient().get_secret("github_token")
    clone_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.exists("project"):
        !git clone --depth 1 {clone_url} project
    os.chdir("project/main" if os.path.exists("project/main") else "project")
    print(f"CWD: {os.getcwd()}")
except Exception as e:
    print(f"Clone failed: {e}. Defaulting to current dir.")

# Link Data
KAGGLE_INPUT = "/kaggle/input/vnexpress-data"
if os.path.exists(KAGGLE_INPUT) and not os.path.exists("data/raw"):
    os.makedirs("data", exist_ok=True)
    !ln -s {KAGGLE_INPUT} data/raw

In [ ]:
# --- GOLD STANDARD HYPERPARAMETERS ---
EPOCHS = 100
LR = 0.001
EMB_DIM = 64
LAYERS = 3
BATCH_SIZE = 2048
GRAPH_TYPE = 'strict_g2'
DATA_PATH = 'data/processed'

# Evaluation Protocols to run
PROTOCOLS = ['full', 'loo100', 'cold']

print(f"Hyperparameters: EPOCHS={EPOCHS}, LR={LR}, DIM={EMB_DIM}")
print(f"Protocols to evaluate: {PROTOCOLS}")

---
## 🛠️ Step 1: Data Preparation
Generate the graph variants used in the experiments.

In [ ]:
# Strict G2 (Primary benchmark for presentation)
!python src/data/convert_to_gnn.py --graph-type hetero --output data/processed/strict_g2 --min-user-interactions 2 --min-article-interactions 2

# G3 (Category Hubs) - Step 1: Base Hetero
!python src/data/convert_to_gnn.py --graph-type hetero --output data/processed/strict_g3 --min-user-interactions 2 --min-article-interactions 2
# G3 (Category Hubs) - Step 2: Augment with Categories
!python src/data/augment_graph_with_categories.py --input-dir data/processed/strict_g3 --output-dir data/processed/strict_g3

---
## 🧪 Experiment A: Content-Based (Semantic Only)
These models ignore the graph and only look at the article text.

In [ ]:
# 1. Generate Embeddings first
!python scripts/generate_multi_embeddings.py --model all --data-path $DATA_PATH/$GRAPH_TYPE

In [ ]:
# 2. Benchmark Content-Based Performance across protocols
EMBEDDINGS = ["tfidf", "phobert", "vn-sbert", "bge-m3", "e5-base"]
for p in PROTOCOLS:
    for emb in EMBEDDINGS:
        print(f"\n{'='*40}\nEvaluating CB: {emb} | Protocol: {p}\n{'='*40}")
        !python scripts/benchmark_cbf.py --model {emb} --data-path $DATA_PATH/$GRAPH_TYPE --eval-protocol {p} --output results/cb_{emb}_{p}.json

---
## 🧬 Experiment B: Collaborative Filtering (Behavioral & Graph)
These models use GNNs and Contrastive Learning to learn from the network.

In [ ]:
# 1. GNN Model Comparison (on Strict G2) across protocols
MODELS = ["simgcl", "xsimgcl", "lightgcl", "ma-hcl", "sim-mahgn"]
for p in PROTOCOLS:
    for m in MODELS:
        print(f"\n{'='*40}\nTraining CF: {m} | Protocol: {p}\n{'='*40}")
        !python scripts/train_cf_models.py --model {m} --epochs $EPOCHS --lr $LR --hidden-dim $EMB_DIM --data-path $DATA_PATH/$GRAPH_TYPE --eval-protocol {p} --save-results results/cf_{m}_{p}.json

---
## 📊 Results Summary

In [ ]:
import json, glob, os
import pandas as pd
results = []
for f in glob.glob("results/*.json"):
    try:
        with open(f) as fp:
            data = json.load(fp)
            # Extract model name and protocol from filename
            fname = os.path.basename(f).replace('.json', '')
            parts = fname.split('_')
            model_type = parts[0].upper() if len(parts) > 0 else 'CF'
            model_name = parts[1].upper() if len(parts) > 1 else fname
            protocol = parts[2] if len(parts) > 2 else 'full'
            
            results.append({
                "Model": f"{model_type}-{model_name}", 
                "Protocol": protocol,
                "Recall@10": data.get('recall@10', 0), 
                "NDCG@10": data.get('ndcg@10', 0)
            })
    except: pass
if results:
    df = pd.DataFrame(results).sort_values(['Protocol', 'Recall@10'], ascending=[True, False])
    display(df)
else:
    print("No results found in results/ folder.")

In [ ]:
# Zip and Download Weights
!zip -r training_output.zip models/ results/
from IPython.display import FileLink
display(FileLink(r'training_output.zip'))